# TB Portals — Example CXRs Panel (Figure F4)
Selects 1 healthy + 1 severe case per country from the TB Portals manifest and saves the raw CXR images for inclusion in the paper's dataset section.

Attach: `tb-portals-cxr-pngs`. Internet **ON**. CPU only. Runtime ≈ 2 min.

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
print('ready')

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

WORK = '/kaggle/working'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'

import sys
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df['timika'] = paper_df['alp_0_100'] + 40 * paper_df['cavity']
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest:', len(paper_df))

In [ ]:
OUT_DIR = Path(f'{WORK}/example_cxrs')
OUT_DIR.mkdir(exist_ok=True)

rows = []
fig, axes = plt.subplots(3, 2, figsize=(5.5, 7.5))
for r, country in enumerate(['Romania', 'Moldova', 'Kazakhstan']):
    sub = paper_df[paper_df['country'] == country].copy()
    # healthy: lowest Timika (with image exists)
    healthy = sub[sub['timika'] < 5].sample(1, random_state=42)
    # severe: highest Timika
    severe = sub.nlargest(20, 'timika').sample(1, random_state=42)
    for c, (tag, row) in enumerate([('healthy', healthy.iloc[0]), ('severe', severe.iloc[0])]):
        img = np.asarray(Image.open(row['image_path']).convert('L'), dtype=np.float32) / 255.0
        ax = axes[r, c]
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'{country} {tag}: Timika={int(row["timika"])}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        rows.append({'country': country, 'tag': tag, 'image_id': row['image_id'],
                     'timika': int(row['timika']), 'image_path': row['image_path']})
plt.tight_layout(pad=0.4)
panel_path = OUT_DIR / 'example_cxrs_panel.pdf'
plt.savefig(panel_path, bbox_inches='tight')
panel_png = OUT_DIR / 'example_cxrs_panel.png'
plt.savefig(panel_png, dpi=140, bbox_inches='tight')
plt.close()

pd.DataFrame(rows).to_csv(OUT_DIR / 'meta.csv', index=False)
import shutil
zip_path = shutil.make_archive(f'{WORK}/example_cxrs', 'zip', str(OUT_DIR))
print('panel ->', panel_path)
print('zip ->', zip_path)